In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .config("spark.jars","C:\\Users\\manpr\\Desktop\\Work\\Requires\\postgresql-42.7.8.jar")\
        .appName("Experiment")\
        .getOrCreate()

jdbc_url = "jdbc:postgresql://zodiac-rds.choegu2w8qpt.ap-south-1.rds.amazonaws.com:5432/zodiac"

jdbc_prop = {
    "user" : "postgres",
    "password" : "6*HT!99052y628a85a2-9474-475a-b234-f00bbae11964",
}

df = spark.read.jdbc(url=jdbc_url,table="BlinkitBrand",properties=jdbc_prop)


KeyboardInterrupt: 

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Parameters
num_days = 365  # one year of data
num_stores = 10
num_products = 20
start_date = datetime(2024, 1, 1)

# Generate date range
dates = [start_date + timedelta(days=i) for i in range(num_days)]

data = []

for store in range(1, num_stores + 1):
    for product in range(1, num_products + 1):
        base_sales = np.random.randint(500, 2000)  # base sales per day
        for date in dates:
            # Add weekday effect (higher sales on weekends)
            weekday = date.weekday()
            weekday_factor = 1.2 if weekday >= 5 else 1.0
            
            # Add seasonality effect (e.g., higher sales in December)
            month_factor = 1.0 + (0.3 if date.month in [11, 12] else 0)
            
            # Add random noise
            noise = np.random.normal(0, 100)
            
            # Compute normal sales
            sales = base_sales * weekday_factor * month_factor + noise
            
            # Random discount and impressions
            discount = np.random.choice([0, 5, 10, 15, 20])
            impressions = int(abs(np.random.normal(10000, 3000)))
            
            # Introduce anomalies (5% chance)
            is_anomaly = 0
            if np.random.rand() < 0.05:
                anomaly_type = random.choice(['spike', 'drop', 'zero'])
                is_anomaly = 1
                if anomaly_type == 'spike':
                    sales *= np.random.uniform(2, 5)  # big spike
                elif anomaly_type == 'drop':
                    sales *= np.random.uniform(0.1, 0.5)  # sudden drop
                elif anomaly_type == 'zero':
                    sales = 0  # no sales
                
            data.append([
                date, store, product, round(max(sales, 0), 2),
                is_anomaly, impressions, discount, weekday
            ])

# Create DataFrame
df = pd.DataFrame(data, columns=[
    "date", "store_id", "product_id", "sales",
    "is_anomaly", "impressions", "discount(%)", "day_of_week"
])

# Shuffle rows
df = df.sample(frac=1).reset_index(drop=True)

# Save dataset
df.to_csv("sales_anomaly_dataset.csv", index=False)
print("✅ Sales anomaly dataset created: sales_anomaly_dataset.csv")
print(df.head(10))


✅ Sales anomaly dataset created: sales_anomaly_dataset.csv
        date  store_id  product_id    sales  is_anomaly  impressions  \
0 2024-04-08         9          18   544.90           0         7890   
1 2024-02-24         1          17   873.78           0        11909   
2 2024-09-14         6           4  1436.41           0        15766   
3 2024-05-17         4           6  1314.15           0        10104   
4 2024-09-29         8          12  1878.18           0         7236   
5 2024-02-09         9          12  1711.53           0         7853   
6 2024-11-18         3          11  1831.84           0         3213   
7 2024-06-15         4           3  1453.23           0        12332   
8 2024-08-26        10           6  1116.54           0        12885   
9 2024-04-08         1          12  1252.04           0         9475   

   discount(%)  day_of_week  
0            0            0  
1           10            5  
2            5            5  
3            5            4 